In [1]:
# 필요한 라이브러리 설치
!pip install streamlit
!pip install torch
!pip install transformers
!pip install sentence-transformers
!pip install qdrant-client
!pip install langchain
!pip install langchain-community
!pip install openai
!pip install python-dotenv


In [2]:
import streamlit as st
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Qdrant
from langchain.schema import Document
from openai import OpenAI
import logging
import time
import datetime
from typing import List, Dict, Any, Optional, Tuple

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# API 키 및 URL 설정 함수
def set_api_keys():
    if 'OPENAI_API_KEY' not in st.session_state:
        st.session_state.OPENAI_API_KEY = st.text_input("OpenAI API 키를 입력하세요:", type="password")
    if 'QDRANT_URL' not in st.session_state:
        st.session_state.QDRANT_URL = st.text_input("Qdrant URL을 입력하세요:")
    if 'QDRANT_API_KEY' not in st.session_state:
        st.session_state.QDRANT_API_KEY = st.text_input("Qdrant API 키를 입력하세요:", type="password")

    return (st.session_state.OPENAI_API_KEY,
            st.session_state.QDRANT_URL,
            st.session_state.QDRANT_API_KEY)

# 전역 변수 및 클라이언트 초기화
COLLECTION_NAME = "son99_d"

# 사용량 설정
MAX_GPT_USAGE = 3
gpt_usage_count = 0
last_reset_date = datetime.date.today()

# 모델 초기화 및 로드
@st.cache_resource
def load_model():
    model_name = "centwon/ko-gpt-trinity-1.2B-v0.5_v3"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to('cuda' if torch.cuda.is_available() else 'cpu')
    return tokenizer, model

tokenizer, model = load_model()

# Qdrant 검색 클래스
class QdrantRetriever:
    def __init__(self, client: QdrantClient, collection_name: str):
        self.client = client
        self.collection_name = collection_name

    def retrieve(self, query_vector: List[float], top_k: int = 10) -> List[Document]:
        results = self.client.search(
            collection_name=self.collection_name,
            query_vector=query_vector,
            limit=top_k
        )
        documents = []
        for result in results:
            if result.score >= 0.7:
                metadata = result.payload
                full_metadata = {
                    '질병': metadata.get('질병', 'N/A'),
                    '의도': metadata.get('의도', 'N/A'),
                    '유사도': result.score,
                    '답변': metadata.get('답변', 'N/A'),
                }
                documents.append(Document(
                    page_content=metadata.get('답변', 'N/A'),
                    metadata=full_metadata
                ))
        return documents

# 간단한 검색 함수
def simple_search(query: str, top_k: int = 5) -> List[Document]:
    query_vector = embeddings.embed_query(query)
    retriever = QdrantRetriever(client=client, collection_name=COLLECTION_NAME)
    return retriever.retrieve(query_vector, top_k)

# GPT-4 응답 생성 함수
def generate_gpt4_response(query: str, context: Optional[str] = None, metadata: Optional[Dict[str, str]] = None) -> str:
    try:
        system_message = (
            "You are an expert medical assistant designed to provide accurate and friendly medical advice to seniors in emergency situations. "
            "Your role is to deliver reliable information based on users' questions about diseases or symptoms, "
            "and to offer appropriate advice based on comprehensive information about the diseases. "
            "Your responses should include details on the prevention, causes, definition, symptoms, diagnosis, and treatment of the disease, "
            "selecting the relevant information according to the user's query. "
            "Please explain everything in a clear and accessible manner, ensuring that the information is tailored for senior users. "
            "Responses should be coherent and continuous, effectively delivering the necessary information. "
            "Additionally, if there is a token limit, ensure that the response is completed within that limit."
        )
        user_message = (
            f"User Query: {query}\n\n"
            f"Context: {context}\n\n"
            f"- Disease: {metadata.get('질병', 'N/A') if metadata else 'N/A'}\n"
            f"- Reference Answer: {metadata.get('답변', 'N/A') if metadata else 'N/A'}"
        )
        response = openai_client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message}
            ],
            temperature=0.5,
            n=1,
            stream=False,
            presence_penalty=1.0
        )

        if metadata:
            logging.info('-'*30)
            logging.info('gpt 모델 사용중')
            logging.info('답변에 관한 정보입니다')
            logging.info('-'*30)
            logging.info(f" 질병 : {metadata.get('질병', 'N/A')}")
            logging.info(f" 의도 : {metadata.get('의도', 'N/A')}")
            logging.info(f"참고한 답변 : {metadata.get('답변', 'N/A')}")
            logging.info(f"유사도 : {metadata.get('유사도', 'N/A')}")

        generated_response = response.choices[0].message.content.strip()
        return generated_response

    except Exception as e:
        logging.error(f"GPT-4 응답 생성 중 오류 발생: {str(e)}", exc_info=True)
        return "죄송합니다. GPT-4 응답을 생성하는 중에 오류가 발생했습니다."

# 개선된 커스텀 모델 응답 생성 함수
def generate_custom_model_response(query: str, context: Optional[str] = None, metadata: Optional[Dict[str, str]] = None, max_tokens: int = 200) -> str:
    try:
        system_message = (
            "You are an expert medical assistant providing accurate and friendly advice to seniors. "
            "Focus on answering the user's query concisely and clearly, using simple language. "
            "If asked about a specific disease, briefly mention its definition, main symptoms, and basic treatment options. "
            "Always prioritize the most relevant information to the user's question."
        )

        user_message = (
            f"Question: {query}\n\n"
            f"Related Context: {context or 'N/A'}\n\n"
            f"Disease: {metadata.get('질병', 'N/A') if metadata else 'N/A'}\n"
            f"Reference: {metadata.get('답변', 'N/A') if metadata else 'N/A'}"
        )

        full_prompt = f"{system_message}\n\n{user_message}\n\nAnswer:"
        input_ids = tokenizer.encode(full_prompt, return_tensors='pt').to(model.device)

        response = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_tokens,
            num_return_sequences=1,
            no_repeat_ngram_size=3,
            temperature=0.7,
            top_k=50,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

        response_text = tokenizer.decode(response[0], skip_special_tokens=True).strip()
        response_text = response_text.split("Answer:")[-1].strip()

        if metadata:
            logging.info('-'*30)
            logging.info('파인튜닝 모델 사용중')
            logging.info('답변에 관한 정보입니다')
            logging.info('-'*30)
            logging.info(f" 질병 : {metadata.get('질병', 'N/A')}")
            logging.info(f" 의도 : {metadata.get('의도', 'N/A')}")
            logging.info(f"참고한 답변 : {metadata.get('답변', 'N/A')}")
            logging.info(f"유사도 : {metadata.get('유사도', 'N/A')}")

        return response_text

    except Exception as e:
        logging.error(f"응답 생성 중 오류 발생: {str(e)}", exc_info=True)
        return "죄송합니다. 응답을 생성하는 중에 오류가 발생했습니다."

# 응답 생성 함수
def generate_response(query: str, context: Optional[str] = None, metadata: Optional[Dict[str, str]] = None) -> Tuple[str, str]:
    global gpt_usage_count
    try:
        if gpt_usage_count < MAX_GPT_USAGE:
            return generate_gpt4_response(query, context, metadata), "GPT-4"
        else:
            return generate_custom_model_response(query, context, metadata), "Custom Model"

    except Exception as e:
        logging.error(f"응답 생성 중 오류 발생: {str(e)}", exc_info=True)
        return "죄송합니다. 응답을 생성하는 중에 오류가 발생했습니다.", "Error"

# 쿼리 처리 함수
def process_query(query: str, chat_history: List[Dict[str, str]]) -> Tuple[str, str, float]:
    start_time = time.time()
    logging.info(f" 질문 : {query}")
    try:
        search_results = simple_search(query, 5)
        best_match = search_results[0]
        context = best_match.page_content

        response, model = generate_response(query, context, best_match.metadata)

        if best_match.metadata:
            logging.info('-'*30)
            logging.info('참고한 메타데이터 정보')
            logging.info('-'*30)
            logging.info(f" 질병 : {best_match.metadata.get('질병', 'N/A')}")
            logging.info(f" 의도 : {best_match.metadata.get('의도', 'N/A')}")
            logging.info(f"참고한 답변 : {best_match.metadata.get('답변', 'N/A')}")
            logging.info(f"유사도 : {best_match.metadata.get('유사도', 'N/A')}")

        processing_time = time.time() - start_time
        logging.info(f"처리 시간: {processing_time:.2f}초")

        return response, model, processing_time

    except Exception as e:
        logging.error(f"쿼리 처리 중 오류 발생: {str(e)}", exc_info=True)
        return "죄송합니다. 응답을 생성하는 중에 오류가 발생했습니다.", "Error", time.time() - start_time

# Streamlit 메인 함수
def main():
    st.title("의료 정보 챗봇")

    # API 키 설정
    OPENAI_API_KEY, QDRANT_URL, QDRANT_API_KEY = set_api_keys()

    if OPENAI_API_KEY and QDRANT_URL and QDRANT_API_KEY:
        # 클라이언트 초기화
        global client, embeddings, openai_client
        client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
        embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
        openai_client = OpenAI(api_key=OPENAI_API_KEY)

        # 챗봇 로직
        if 'chat_history' not in st.session_state:
            st.session_state.chat_history = []

        if 'model_selected' not in st.session_state:
            st.session_state.model_selected = False
            st.session_state.chat_history.append({"role": "assistant", "content": "모델을 골라주세요."})

        for chat in st.session_state.chat_history:
            with st.chat_message(chat["role"]):
                st.write(chat["content"])

        if not st.session_state.model_selected:
            col1, col2, col3 = st.columns(3)

            with col1:
                if st.button("GPT-4 모델 사용 (3회 제한)"):
                    st.session_state.model_selected = "GPT-4"
                    st.session_state.chat_history.append({"role": "assistant", "content": "GPT-4 모델을 선택하셨습니다."})

            with col2:
                if st.button("커스텀 모델 사용 (토큰 200)"):
                    st.session_state.model_selected = "Custom-200"
                    st.session_state.chat_history.append({"role": "assistant", "content": "커스텀 모델(토큰 200)을 선택하셨습니다."})

            with col3:
                if st.button("커스텀 모델 사용 (토큰 800)"):
                    st.session_state.model_selected = "Custom-800"
                    st.session_state.chat_history.append({"role": "assistant", "content": "커스텀 모델(토큰 800)을 선택하셨습니다."})

        if st.session_state.model_selected:
            user_input = st.chat_input("질문을 입력하세요...")

            if user_input:
                st.session_state.chat_history.append({"role": "user", "content": user_input})
                with st.chat_message("user"):
                    st.write(user_input)

                with st.spinner("답변 생성 중..."):
                    response, model, processing_time = process_query(user_input, st.session_state.chat_history)
                    st.session_state.chat_history.append({"role": "assistant", "content": response})

                with st.chat_message("assistant"):
                    st.write(response)
                    st.write(f"모델: {model}, 처리 시간: {processing_time:.2f}초")

                # GPT-4 사용 횟수 업데이트
                if model == "GPT-4":
                    global gpt_usage_count
                    gpt_usage_count += 1
                    if gpt_usage_count >= MAX_GPT_USAGE:
                        st.warning("GPT-4 모델 사용 횟수를 모두 소진했습니다. 이제 커스텀 모델을 사용합니다.")

    else:
        st.warning("모든 API 키와 URL을 입력해주세요.")

if __name__ == "__main__":
    main()

2024-08-28 12:56:20.297 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-08-28 12:56:20.812 
  command:

    streamlit run /usr/local/lib/python3.10/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2024-08-28 12:56:20.821 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-08-28 12:56:20.825 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-08-28 12:56:21.332 Thread 'Thread-10': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-08-28 12:56:21.335 Thread 'Thread-10': missing ScriptRunContext! This warning can be ignored when running in bare mode.
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the 

tokenizer_config.json:   0%|          | 0.00/64.0k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/732k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/233k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/577 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/868 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

2024-08-28 12:57:28.185 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-08-28 12:57:28.187 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-08-28 12:57:28.194 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-08-28 12:57:28.197 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-08-28 12:57:28.198 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-08-28 12:57:28.201 Session state does not function when running a script without `streamlit run`
2024-08-28 12:57:28.203 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-08-28 12:57:28.206 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-08-28 12:57